# US Stock Monthly Market Cap Collection
**목적:** FMP API에서 미국 주식 월별 시가총액 데이터를 수집하여 DB에 저장

| Cell | 내용 |
|------|------|
| Cell 1 | 라이브러리 임포트 & 환경 설정 |
| Cell 2 | DB 연결 확인 및 테이블 생성 |
| Cell 3 | DB에서 indicator 값 확인 (시가총액 컬럼명 탐색) |
| Cell 4 | FMP API 단일 티커 테스트 |
| Cell 5 | 증분 수집 기준일 확인 함수 |
| Cell 6 | 전체 티커 수집 & DB 저장 (메인 실행) |
| Cell 7 | 저장 결과 검증 |

---
## Cell 1 — 라이브러리 임포트 & 환경 설정

In [1]:
import sys
import os
import time
import requests
import pandas as pd
import pymysql
from datetime import datetime, date
from typing import Optional, List

# ── 환경별 DATA 폴더 경로 자동 감지 ──────────────────────────────────
DATA_PATHS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA",
]

DATA_DIR = None
for _p in DATA_PATHS:
    if os.path.exists(_p):
        DATA_DIR = _p
        break

if DATA_DIR is None:
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")

# ── DATA 폴더와 그 상위 폴더를 모두 sys.path에 추가 ──────────────────
PARENT_DIR = os.path.dirname(DATA_DIR)   # DATA 상위 폴더

for _path in [DATA_DIR, PARENT_DIR]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

print(f"✅ DATA 폴더  : {DATA_DIR}")
print(f"✅ 상위 폴더  : {PARENT_DIR}")
print(f"✅ sys.path 앞: {sys.path[:3]}")

# ── config & ticker list 임포트 ───────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list

print(f"✅ config 임포트 성공")
print(f"✅ 티커 리스트 로드: {len(ticker_list)}개")

# ── FMP API 설정 ──────────────────────────────────────────────────────
FMP_API_KEY = "여기에_FMP_API_KEY_입력"
FMP_BASE_URL = "https://financialmodelingprep.com/api/v3"

RATE_LIMIT_PER_MIN  = 240
RATE_LIMIT_INTERVAL = 60 / RATE_LIMIT_PER_MIN

TARGET_TABLE = "us_stock_daily_market_cap"

print(f"\n⚙️  Rate Limit: {RATE_LIMIT_PER_MIN}콜/분 (간격 {RATE_LIMIT_INTERVAL:.3f}초)")
print(f"⚙️  저장 테이블: {TARGET_TABLE}")

✅ DATA 폴더  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✅ 상위 폴더  : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
✅ sys.path 앞: ['C:\\Users\\82108\\OneDrive\\바탕 화면\\investment\\investment_strategy\\DATA', 'C:\\Users\\82108\\OneDrive\\바탕 화면\\investment\\investment_strategy\\US_Market\\collect\\us_stock_data\\stock_price_and_marketcap', 'C:\\Users\\82108\\OneDrive\\바탕 화면\\investment\\investment_strategy']
✅ config 임포트 성공
✅ 티커 리스트 로드: 2000개

⚙️  Rate Limit: 240콜/분 (간격 0.250초)
⚙️  저장 테이블: us_stock_daily_market_cap


---
## Cell 2 — DB 연결 확인 및 테이블 생성

In [2]:
def get_connection():
    """pymysql 커넥션 반환"""
    db_info = get_db_info()
    return pymysql.connect(
        host=db_info['host'],
        port=int(db_info['port']),   # ← 추가 (3307)
        user=db_info['user'],
        password=db_info['password'],
        database=db_info['database'],
        charset='utf8mb4',
        autocommit=False
    )

# ── 연결 테스트 ────────────────────────────────────────────────────────
conn_test = get_connection()
print("✅ DB 연결 성공")
conn_test.close()

# ── 테이블 생성 (없으면 자동 생성) ────────────────────────────────────
CREATE_TABLE_SQL = f"""
CREATE TABLE IF NOT EXISTS `{TARGET_TABLE}` (
    id            BIGINT       NOT NULL AUTO_INCREMENT,
    date          DATE         NOT NULL COMMENT '데이터 기준일(월말)',
    ticker        VARCHAR(20)  NOT NULL COMMENT '티커 심볼',
    indicator     VARCHAR(50)  NOT NULL COMMENT '지표명 (market_cap)',
    value         DOUBLE       COMMENT '시가총액(USD)',
    collected_at  DATETIME     NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT '수집 시각',
    PRIMARY KEY (id),
    UNIQUE KEY uq_date_ticker_indicator (date, ticker, indicator),
    INDEX idx_ticker (ticker),
    INDEX idx_date   (date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
COMMENT='미국 주식 월별 시가총액 (long-format)';
"""

conn = get_connection()
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_TABLE_SQL)
    conn.commit()
    print(f"✅ 테이블 '{TARGET_TABLE}' 준비 완료 (없으면 생성, 있으면 유지)")
finally:
    conn.close()

✅ DB 연결 성공
✅ 테이블 'us_stock_daily_market_cap' 준비 완료 (없으면 생성, 있으면 유지)


---
## Cell 3 — DB indicator 값 확인
기존 테이블에서 시가총액이 어떤 indicator 값으로 저장되어 있는지 탐색합니다.

In [3]:
# ── 기존 DB 테이블에서 indicator 종류 탐색 ─────────────────────────────
# 시가총액 관련 테이블이 있다면 어떤 indicator 값으로 저장됐는지 확인

conn = get_connection()
try:
    with conn.cursor() as cur:
        
        # 1) 현재 DB의 테이블 목록 확인
        cur.execute("SHOW TABLES")
        rows = cur.fetchall()
        tables = [r[0] for r in rows]
        print("📋 DB 테이블 목록:")
        for t in tables:
            print(f"   - {t}")
        
        print()
        
        # 2) 'market' 또는 'cap' 키워드가 포함된 테이블 탐색
        market_cap_tables = [t for t in tables if 'market' in t.lower() or 'cap' in t.lower()]
        if market_cap_tables:
            print(f"🔍 시가총액 관련 테이블 후보: {market_cap_tables}")
            for tbl in market_cap_tables:
                cur.execute(f"SELECT DISTINCT indicator FROM `{tbl}` LIMIT 20")
                inds = cur.fetchall()
                print(f"   [{tbl}] indicators: {[r[0] for r in inds]}")
        else:
            print("⚠️  시가총액 관련 기존 테이블 없음 → 새로 생성 예정")
        
        print()
        
        # 3) 주가 데이터 테이블에서 indicator 목록 확인 (참고용)
        price_tables = [t for t in tables if 'price' in t.lower() or 'stock' in t.lower() or 'daily' in t.lower()]
        if price_tables:
            print(f"📈 주가 관련 테이블: {price_tables}")
            for tbl in price_tables[:3]:  # 최대 3개만
                try:
                    cur.execute(f"SELECT DISTINCT indicator FROM `{tbl}` LIMIT 30")
                    inds = cur.fetchall()
                    print(f"   [{tbl}] indicators: {[r[0] for r in inds]}")
                except Exception as e:
                    print(f"   [{tbl}] 조회 실패: {e}")
        
finally:
    conn.close()

# ── 이 노트북에서 사용할 indicator 값 결정 ────────────────────────────
MARKET_CAP_INDICATOR = "market_cap"   # ← 위 조회 결과 보고 필요시 수정
print(f"\n✅ 사용할 indicator 값: '{MARKET_CAP_INDICATOR}'")

📋 DB 테이블 목록:
   - Credit_Spread
   - KSE_Price
   - KTB_Yield_Daily
   - KTB_daily
   - Korea_Economy_Data
   - Korea_company_valuation_ver2
   - US_BS_from_FMP
   - US_CF_from_FMP
   - US_IS_from_FMP
   - US_company_valuation_result
   - US_funda
   - US_fundm
   - US_fundq
   - bok_economic_indicators
   - correlation_price_hscode
   - correlation_table_DB
   - firm_char
   - fmp_financial_data
   - hs_code_by_kr_monster_company
   - korea_company_hscode_map
   - korea_company_valuation_result
   - korea_fs_data
   - korea_fs_data_from_DART
   - korea_fs_data_from_DART_V2
   - korea_fs_data_from_DG
   - korea_fs_data_roe_roa
   - korea_monthly_trade_act_forecast_data
   - korea_monthly_trade_data
   - korea_monthly_trade_data_forecast
   - korea_monthly_trade_data_v2
   - korea_monthly_trade_forecast_v2
   - korea_quarterly_trade_data
   - korea_required_return_result
   - korea_revenue_forecast_result
   - ks_listed_company_daily_marketcap
   - price_table
   - psr_data
   - sec_fin

---
## Cell 4 — FMP API 단일 티커 테스트
전체 실행 전, 단일 티커로 API 응답 구조를 확인합니다.

In [4]:
def fetch_market_cap_fmp(ticker: str, from_date: Optional[str] = None) -> pd.DataFrame:
    """
    FMP API에서 특정 티커의 월별 시가총액 데이터를 수집.
    
    Parameters
    ----------
    ticker    : 티커 심볼 (예: 'AAPL')
    from_date : 수집 시작일 'YYYY-MM-DD' (None이면 전체 기간)
    
    Returns
    -------
    DataFrame with columns: [date, ticker, indicator, value]
    """
    FMP_API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
    url = f"{FMP_BASE_URL}/historical-market-capitalization/{ticker}"
    params = {
        "apikey": FMP_API_KEY,
        "limit": 600,   # 최대 600개 (약 50년치 월별)
    }
    if from_date:
        params["from"] = from_date
    
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
    except requests.exceptions.RequestException as e:
        print(f"  ⚠️  API 오류 [{ticker}]: {e}")
        return pd.DataFrame()
    
    if not data or not isinstance(data, list):
        return pd.DataFrame()
    
    df = pd.DataFrame(data)
    
    # FMP 응답 컬럼 확인 후 정규화
    # 예상 컬럼: ['symbol', 'date', 'marketCap']
    if 'date' not in df.columns:
        print(f"  ⚠️  'date' 컬럼 없음 [{ticker}]: {df.columns.tolist()}")
        return pd.DataFrame()
    
    # marketCap 컬럼명 탐색 (FMP 버전에 따라 다를 수 있음)
    cap_col = None
    for candidate in ['marketCap', 'market_cap', 'MarketCap', 'capitalisation']:
        if candidate in df.columns:
            cap_col = candidate
            break
    
    if cap_col is None:
        print(f"  ⚠️  시가총액 컬럼 없음 [{ticker}]: {df.columns.tolist()}")
        return pd.DataFrame()
    
    # ── 월별로 리샘플: 각 월의 마지막 데이터만 유지 ─────────────────
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    df = df.groupby('year_month', as_index=False).last()   # 월말 기준
    
    # ── long-format 변환 ───────────────────────────────────────────────
    result = pd.DataFrame({
        'date'      : df['date'].dt.date,
        'ticker'    : ticker,
        'indicator' : MARKET_CAP_INDICATOR,
        'value'     : pd.to_numeric(df[cap_col], errors='coerce'),
    })
    result = result.dropna(subset=['value'])
    result = result[result['value'] > 0]
    
    return result


# ── 단일 티커 테스트 ──────────────────────────────────────────────────
TEST_TICKER = "AAPL"
print(f"🧪 테스트 티커: {TEST_TICKER}")
print(f"   API URL: {FMP_BASE_URL}/historical-market-capitalization/{TEST_TICKER}")

df_test = fetch_market_cap_fmp(TEST_TICKER)

if df_test.empty:
    print("❌ 데이터 없음 — API Key 및 티커 확인 필요")
else:
    print(f"\n✅ 수집 성공: {len(df_test)}개 레코드")
    print(f"   기간: {df_test['date'].min()} ~ {df_test['date'].max()}")
    print(f"\n{df_test.head(8).to_string(index=False)}")
    print(f"\n{df_test.tail(5).to_string(index=False)}")
    print(f"\n📊 컬럼: {df_test.dtypes.to_dict()}")

🧪 테스트 티커: AAPL
   API URL: https://financialmodelingprep.com/api/v3/historical-market-capitalization/AAPL

✅ 수집 성공: 30개 레코드
   기간: 2023-11-30 ~ 2026-04-02

      date ticker  indicator         value
2023-11-30   AAPL market_cap 2946079481850
2023-12-29   AAPL market_cap 2986094670390
2024-01-31   AAPL market_cap 2840839846400
2024-02-29   AAPL market_cap 2784608472000
2024-03-28   AAPL market_cap 2641796186880
2024-04-30   AAPL market_cap 2603923451930
2024-05-31   AAPL market_cap 2939025912250
2024-06-28   AAPL market_cap 3219857673020

      date ticker  indicator         value
2025-12-31   AAPL market_cap 4009434233880
2026-01-30   AAPL market_cap 3826852037840
2026-02-27   AAPL market_cap 3896168380440
2026-03-31   AAPL market_cap 3742935018820
2026-04-02   AAPL market_cap 3774348595360

📊 컬럼: {'date': dtype('O'), 'ticker': dtype('O'), 'indicator': dtype('O'), 'value': dtype('int64')}


---
## Cell 5 — 증분 수집 기준일 확인 함수

In [13]:
def get_last_date_in_db(ticker: str, override_date: Optional[str] = None) -> Optional[str]:
    """
    DB에서 특정 티커의 마지막 저장 날짜를 조회.
    
    Parameters
    ----------
    ticker        : 티커 심볼
    override_date : 'YYYY-MM-DD' 형식으로 직접 지정하면 DB 조회 무시
    
    Returns
    -------
    'YYYY-MM-DD' 문자열 또는 None (전체 수집)
    """
    if override_date is not None:
        print(f"   📅 override 기준일 사용: {override_date}")
        return override_date
    
    conn = get_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(
                f"SELECT MAX(date) FROM `{TARGET_TABLE}` "
                f"WHERE ticker = %s AND indicator = %s",
                (ticker, MARKET_CAP_INDICATOR)
            )
            row = cur.fetchone()
            last_date = row[0] if row and row[0] else None
    finally:
        conn.close()
    
    if last_date is None:
        return None  # 전체 기간 수집
    
    # 마지막 날짜 이후부터 수집 (중복 방지를 위해 +1일)
    from datetime import timedelta
    next_date = (pd.to_datetime(last_date) + timedelta(days=1)).strftime('%Y-%m-%d')
    return next_date


def upsert_to_db(conn, df: pd.DataFrame) -> int:
    """
    DataFrame을 DB에 upsert (중복 시 무시, 신규만 삽입).
    
    Returns
    -------
    삽입된 행 수
    """
    if df.empty:
        return 0
    
    SQL = f"""
        INSERT IGNORE INTO `{TARGET_TABLE}`
            (date, ticker, indicator, value)
        VALUES (%s, %s, %s, %s)
    """
    rows = [
        (str(r['date']), r['ticker'], r['indicator'], float(r['value']))
        for _, r in df.iterrows()
    ]
    
    with conn.cursor() as cur:
        cur.executemany(SQL, rows)
    conn.commit()
    
    return len(rows)


# ── 테스트 ────────────────────────────────────────────────────────────
print("🧪 기준일 조회 테스트 (AAPL)")
last = get_last_date_in_db("AAPL")
if last:
    print(f"   DB 마지막 날짜 이후 수집: {last} ~")
else:
    print("   DB에 데이터 없음 → 전체 기간 수집")

print("\n🧪 override 테스트 (2023-01-01 지정)")
last_override = get_last_date_in_db("AAPL", override_date="2023-01-01")
print(f"   결과: {last_override}")

🧪 기준일 조회 테스트 (AAPL)
   DB에 데이터 없음 → 전체 기간 수집

🧪 override 테스트 (2023-01-01 지정)
   📅 override 기준일 사용: 2023-01-01
   결과: 2023-01-01


---
## Cell 6 — 전체 티커 수집 & DB 저장 (메인 실행)

**파라미터 안내**
| 파라미터 | 설명 | 기본값 |
|---------|------|-------|
| `OVERRIDE_FROM_DATE` | 전체 티커 공통 기준일 강제 지정. `None`이면 티커별 DB 마지막 날짜 자동 사용 | `None` |
| `BATCH_SIZE` | 한 번에 DB에 저장할 티커 수 | `10` |
| `TEST_MODE` | `True`이면 앞 5개 티커만 실행 | `False` |

In [15]:
# ── 실행 파라미터 ─────────────────────────────────────────────────────
OVERRIDE_FROM_DATE = '2015-01-01'      # 예: "2022-01-01" 또는 None (자동)
BATCH_SIZE         = 10      # 티커 묶음 단위로 DB 저장
TEST_MODE          = True    # ← 처음엔 True로 테스트, 전체 실행 시 False

# ─────────────────────────────────────────────────────────────────────
target_tickers = ticker_list[:5] if TEST_MODE else ticker_list
total = len(target_tickers)

print(f"{'🧪 테스트 모드' if TEST_MODE else '🚀 전체 실행'}: {total}개 티커")
if OVERRIDE_FROM_DATE:
    print(f"📅 공통 기준일 override: {OVERRIDE_FROM_DATE}")
else:
    print("📅 기준일: 티커별 DB 마지막 날짜 자동 감지")
print("-" * 60)

# ── 수집 실행 ────────────────────────────────────────────────────────
success_count  = 0
skip_count     = 0
error_count    = 0
total_inserted = 0
error_tickers  = []

batch_buffer: List[pd.DataFrame] = []
last_call_time = 0.0

conn = get_connection()

try:
    for idx, ticker in enumerate(target_tickers, start=1):
        
        # 진행률 표시
        pct = idx / total * 100
        print(f"[{idx:4d}/{total}] ({pct:5.1f}%) {ticker:<8}", end="  ")
        
        # ── Rate Limit 처리 ────────────────────────────────────────────
        elapsed = time.time() - last_call_time
        if elapsed < RATE_LIMIT_INTERVAL:
            time.sleep(RATE_LIMIT_INTERVAL - elapsed)
        
        # ── 증분 기준일 결정 ───────────────────────────────────────────
        from_date = get_last_date_in_db(ticker, override_date=OVERRIDE_FROM_DATE)
        
        # ── FMP API 호출 ───────────────────────────────────────────────
        last_call_time = time.time()
        df_ticker = fetch_market_cap_fmp(ticker, from_date=from_date)
        
        if df_ticker.empty:
            print(f"→ 데이터 없음 (skip)")
            skip_count += 1
            continue
        
        # ── 배치 버퍼에 추가 ───────────────────────────────────────────
        batch_buffer.append(df_ticker)
        print(f"→ {len(df_ticker):3d}건 수집", end="")
        
        # ── 배치 단위로 DB 저장 ────────────────────────────────────────
        if len(batch_buffer) >= BATCH_SIZE:
            batch_df = pd.concat(batch_buffer, ignore_index=True)
            inserted = upsert_to_db(conn, batch_df)
            total_inserted += inserted
            print(f"  💾 DB저장 {inserted}건")
            batch_buffer.clear()
        else:
            print()
        
        success_count += 1
        
    # ── 남은 배치 처리 ────────────────────────────────────────────────
    if batch_buffer:
        batch_df = pd.concat(batch_buffer, ignore_index=True)
        inserted = upsert_to_db(conn, batch_df)
        total_inserted += inserted
        print(f"\n💾 잔여 배치 DB저장: {inserted}건")
        batch_buffer.clear()

except Exception as e:
    print(f"\n❌ 예외 발생: {e}")
    import traceback
    traceback.print_exc()
finally:
    conn.close()

# ── 결과 요약 ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("📊 수집 완료 요약")
print(f"   성공   : {success_count:,}개 티커")
print(f"   스킵   : {skip_count:,}개 티커 (데이터 없음)")
print(f"   오류   : {error_count:,}개 티커")
print(f"   DB 저장: {total_inserted:,}건")
if error_tickers:
    print(f"   오류 티커: {error_tickers}")
print("=" * 60)

🧪 테스트 모드: 5개 티커
📅 공통 기준일 override: 2015-01-01
------------------------------------------------------------
[   1/5] ( 20.0%) NVDA         📅 override 기준일 사용: 2015-01-01
→  30건 수집
[   2/5] ( 40.0%) GOOG         📅 override 기준일 사용: 2015-01-01
→  30건 수집
[   3/5] ( 60.0%) AAPL         📅 override 기준일 사용: 2015-01-01
→  30건 수집
[   4/5] ( 80.0%) MSFT         📅 override 기준일 사용: 2015-01-01
→  30건 수집
[   5/5] (100.0%) AMZN         📅 override 기준일 사용: 2015-01-01
→  30건 수집

💾 잔여 배치 DB저장: 150건

📊 수집 완료 요약
   성공   : 5개 티커
   스킵   : 0개 티커 (데이터 없음)
   오류   : 0개 티커
   DB 저장: 150건


---
## Cell 7 — 저장 결과 검증

In [ ]:
conn = get_connection()
try:
    with conn.cursor() as cur:
        
        # 1) 전체 레코드 수
        cur.execute(f"SELECT COUNT(*) FROM `{TARGET_TABLE}`")
        total_rows = cur.fetchone()[0]
        print(f"📊 전체 레코드 수: {total_rows:,}건")
        
        # 2) 티커 수
        cur.execute(f"SELECT COUNT(DISTINCT ticker) FROM `{TARGET_TABLE}`")
        ticker_cnt = cur.fetchone()[0]
        print(f"📊 저장된 티커 수: {ticker_cnt:,}개")
        
        # 3) 날짜 범위
        cur.execute(f"SELECT MIN(date), MAX(date) FROM `{TARGET_TABLE}`")
        min_d, max_d = cur.fetchone()
        print(f"📊 날짜 범위: {min_d} ~ {max_d}")
        
        # 4) indicator 종류
        cur.execute(f"SELECT DISTINCT indicator FROM `{TARGET_TABLE}`")
        indicators = [r[0] for r in cur.fetchall()]
        print(f"📊 indicator 값: {indicators}")
        
        print()
        
        # 5) 샘플 데이터 (AAPL 최근 12개월)
        cur.execute(
            f"SELECT date, ticker, indicator, value "
            f"FROM `{TARGET_TABLE}` "
            f"WHERE ticker = 'AAPL' "
            f"ORDER BY date DESC LIMIT 12"
        )
        rows = cur.fetchall()
        df_check = pd.DataFrame(rows, columns=['date','ticker','indicator','value'])
        df_check['value_조'] = (df_check['value'] / 1e12).round(2).astype(str) + '조$'
        print("🔍 AAPL 최근 12개월 시가총액:")
        print(df_check.to_string(index=False))
        
        print()
        
        # 6) 티커별 데이터 건수 상위 10개
        cur.execute(
            f"SELECT ticker, COUNT(*) as cnt "
            f"FROM `{TARGET_TABLE}` "
            f"GROUP BY ticker "
            f"ORDER BY cnt DESC LIMIT 10"
        )
        rows2 = cur.fetchall()
        df_cnt = pd.DataFrame(rows2, columns=['ticker', '레코드수'])
        print("📋 데이터 건수 상위 10 티커:")
        print(df_cnt.to_string(index=False))

finally:
    conn.close()